# S4 - ML distribuido con Spark MLlib (Regresion)

**Actividad:** integrar tres fuentes reales de sensores ambientales en un DataLake analitico particionado (Bronze -> Silver -> Gold), y sobre esa salida entrenar y comparar modelos de regresion distribuida con Spark MLlib, reportando RMSE, R2 y MAE.

Estructura del notebook: cada fase de CRISP-DM es un bloque (## Fase N), y cada actividad dentro de la fase es un paso numerado (## 3.F.M), igual que en la guia.


## Fase 1 — Business Understanding


## 3.1.1 Crear el notebook y la `SparkSession`


In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("sesion4-ml-distribuido-regresion")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

spark


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/03 02:32:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
ORIGEN_DATOS = "/opt/s04-ml-distribuido-regresion/data"


## 3.1.2 Definir la variable numerica a predecir


**Objetivo:** estimar `Valor_Objetivo` (hoy, `Valor_CE`) a partir de otras variables medidas
en el mismo instante — no un pronostico con historial temporal (eso es contenido de S10, 2.3).


## 3.1.3 Definir la decision de negocio asociada


**Alcance:** comparar un modelo base de regresion lineal, tres configuraciones de
regularizacion y un segundo algoritmo (Random Forest), reportando RMSE, R2 y MAE — sin
busqueda exhaustiva de hiperparametros. El modelo ganador es el que decide si vale la pena
sostener un pipeline de regresion distribuida sobre esta fuente, en vez de no predecir nada.


## Fase 2 — Data Understanding


## 3.2.1 Cargar los datos


In [3]:
from pyspark.sql.types import StructType, StructField, TimestampType, DoubleType

schema_ce = StructType([
    StructField("FechaHora", TimestampType(), nullable=False),
    StructField("Valor_CE", DoubleType(), nullable=True),
])
df_ce = spark.read.csv(f"{ORIGEN_DATOS}/campo_electrico.csv", header=True, schema=schema_ce)

schema_cm = StructType([
    StructField("FechaHora", TimestampType(), nullable=False),
    StructField("Valor_CM", DoubleType(), nullable=True),
])
df_cm = spark.read.csv(f"{ORIGEN_DATOS}/campo_magnetico.csv", header=True, schema=schema_cm)

schema_va = StructType([
    StructField("TempOut", DoubleType(), nullable=True),
    StructField("OutHum", DoubleType(), nullable=True),
    StructField("WindSpeed", DoubleType(), nullable=True),
    StructField("WindDir", DoubleType(), nullable=True),
    StructField("Bar", DoubleType(), nullable=True),
    StructField("Rain", DoubleType(), nullable=True),
    StructField("SolarRad", DoubleType(), nullable=True),
    StructField("UVIndex", DoubleType(), nullable=True),
    StructField("FechaHora", TimestampType(), nullable=False),
])
df_va = spark.read.csv(f"{ORIGEN_DATOS}/variables_ambientales.csv", header=True, schema=schema_va)


**Error frecuente**: la fuente original trae la columna de radiacion solar como `SolarRad.`
(con un punto al final). Referenciarla luego con `col("SolarRad.")` falla con `AnalysisException`
— basta con declarar el nombre ya limpio (`SolarRad`, sin punto) en el `StructField` de arriba,
como ya se hizo en el schema.


`variables_ambientales.csv` trae `FechaHora` duplicada — hay que resolverlo antes de integrar,
porque un `join` contra una clave duplicada multiplica filas del lado que no lo esta:


In [4]:
from pyspark.sql.functions import col, count as spark_count, when, lit
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

columnas_conteo_nulos = [c for c in df_va.columns if c not in ("FechaHora", "WindDir")]

df_va_con_conteo = df_va.withColumn(
    "CantidadNulos",
    sum(when(col(c).isNull(), 1).otherwise(0) for c in columnas_conteo_nulos),
)

ventana_va = Window.partitionBy("FechaHora").orderBy(col("CantidadNulos").asc())

df_va_unico = (
    df_va_con_conteo
    .withColumn("row_num", row_number().over(ventana_va))
    .filter(col("row_num") == 1)
    .drop("row_num", "CantidadNulos")
)


`FechaHora` es la clave comun para integrar; el campo electrico queda como tabla principal
(`left join`), porque interesa el periodo que ese sensor cubre:


In [5]:
df_integrado = (
    df_ce
    .join(df_cm, on="FechaHora", how="left")
    .join(df_va_unico, on="FechaHora", how="left")
)

print(f"Integrado: {df_integrado.count():,} registros x {len(df_integrado.columns)} columnas")


26/09/03 02:32:14 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad., UVIndex, FechaHora
 Schema: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad, UVIndex, FechaHora
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s04-ml-distribuido-regresion/data/variables_ambientales.csv
                                                                                

Integrado: 186,664 registros x 11 columnas


## 3.2.2 Describir las variables


In [6]:
df_integrado.printSchema()

df_integrado.select([
    spark_count(when(col(c).isNull(), c)).alias(c) for c in df_integrado.columns
]).show(vertical=True, truncate=False)


root
 |-- FechaHora: timestamp (nullable = true)
 |-- Valor_CE: double (nullable = true)
 |-- Valor_CM: double (nullable = true)
 |-- TempOut: double (nullable = true)
 |-- OutHum: double (nullable = true)
 |-- WindSpeed: double (nullable = true)
 |-- WindDir: double (nullable = true)
 |-- Bar: double (nullable = true)
 |-- Rain: double (nullable = true)
 |-- SolarRad: double (nullable = true)
 |-- UVIndex: double (nullable = true)



26/09/03 02:32:21 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, WindDir, Bar, Rain, SolarRad., UVIndex, FechaHora
 Schema: TempOut, OutHum, WindSpeed, WindDir, Bar, Rain, SolarRad, UVIndex, FechaHora
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s04-ml-distribuido-regresion/data/variables_ambientales.csv
[Stage 13:===========================================>              (6 + 2) / 8]

-RECORD 0-----------
 FechaHora | 0      
 Valor_CE  | 0      
 Valor_CM  | 0      
 TempOut   | 0      
 OutHum    | 0      
 WindSpeed | 0      
 WindDir   | 186664 
 Bar       | 0      
 Rain      | 0      
 SolarRad  | 0      
 UVIndex   | 0      



## 3.2.3 Analizar estadisticas descriptivas


In [7]:
df_integrado.describe(["Valor_CE", "Valor_CM", "TempOut", "OutHum", "WindSpeed", "Bar", "Rain", "SolarRad", "UVIndex"]).show()


26/09/03 02:32:25 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/09/03 02:32:25 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad., UVIndex, FechaHora
 Schema: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad, UVIndex, FechaHora
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s04-ml-distribuido-regresion/data/variables_ambientales.csv
[Stage 25:>                                                         (0 + 8) / 8]

+-------+-------------------+-----------------+------------------+-----------------+------------------+------------------+--------------------+------------------+------------------+
|summary|           Valor_CE|         Valor_CM|           TempOut|           OutHum|         WindSpeed|               Bar|                Rain|          SolarRad|           UVIndex|
+-------+-------------------+-----------------+------------------+-----------------+------------------+------------------+--------------------+------------------+------------------+
|  count|             186664|           186664|            186664|           186664|            186664|            186664|              186664|            186664|            186664|
|   mean|-0.9511688916984522|25140.69686763381|17.560050143573108|82.29205417220246|3.9206135087643594|  949.189360562269|2.142887755539365E-5|174.59216024514635|1.2261164445206265|
| stddev| 0.8913537022459245|8035.111595596705| 3.006652545768541|7.339985859460037| 4.881

## 3.2.4 Analizar correlacion con el objetivo


In [8]:
predictores_candidatos = [
    c for c in df_integrado.columns
    if c not in ("FechaHora", "Valor_CE", "WindDir")
]

for columna in predictores_candidatos:
    correlacion = df_integrado.stat.corr(columna, "Valor_CE")
    print(f"{columna:12s} correlacion con Valor_CE: {correlacion:.4f}")


26/09/03 02:32:29 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad., UVIndex, FechaHora
 Schema: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad, UVIndex, FechaHora
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s04-ml-distribuido-regresion/data/variables_ambientales.csv
                                                                                

Valor_CM     correlacion con Valor_CE: -0.0249


26/09/03 02:32:32 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad., UVIndex, FechaHora
 Schema: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad, UVIndex, FechaHora
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s04-ml-distribuido-regresion/data/variables_ambientales.csv
                                                                                

TempOut      correlacion con Valor_CE: 0.0715


26/09/03 02:32:35 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad., UVIndex, FechaHora
 Schema: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad, UVIndex, FechaHora
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s04-ml-distribuido-regresion/data/variables_ambientales.csv
                                                                                

...

Rain         correlacion con Valor_CE: -0.0016


26/09/03 02:32:44 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad., UVIndex, FechaHora
 Schema: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad, UVIndex, FechaHora
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s04-ml-distribuido-regresion/data/variables_ambientales.csv
                                                                                

SolarRad     correlacion con Valor_CE: -0.1803


26/09/03 02:32:46 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad., UVIndex, FechaHora
 Schema: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad, UVIndex, FechaHora
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s04-ml-distribuido-regresion/data/variables_ambientales.csv


UVIndex      correlacion con Valor_CE: -0.1601


`WindDir` queda fuera del calculo a proposito: es 100% nula, su correlacion no estaria
definida. Advertencia: todavia no se filtro `Valor_CM = 99999` (3.3.1) ni los nulos sueltos —
esta es una lectura preliminar, se confirma recien con `df_valido` (3.3.2) y con los coeficientes
o `featureImportances` del modelo entrenado (3.4).


## Fase 3 — Data Preparation


## 3.3.1 Limpiar los datos


Dos problemas de calidad distintos, dos tratamientos distintos — ninguno es un nulo comun:
`WindDir` sin valores utiles (columna 100% nula, se descarta completa) y `Valor_CM = 99999`
(codigo de error de sensor, no un nulo — Spark lo ve como un `Double` valido).


In [9]:
nulos_winddir = df_integrado.filter(col("WindDir").isNull()).count()
total = df_integrado.count()
print(f"WindDir nula: {nulos_winddir:,} de {total:,} ({nulos_winddir/total*100:.1f}%)")

df_sin_winddir = df_integrado.drop("WindDir")


26/09/03 02:32:48 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, WindDir, Bar, Rain, SolarRad., UVIndex, FechaHora
 Schema: TempOut, OutHum, WindSpeed, WindDir, Bar, Rain, SolarRad, UVIndex, FechaHora
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s04-ml-distribuido-regresion/data/variables_ambientales.csv
26/09/03 02:32:50 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad., UVIndex, FechaHora
 Schema: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad, UVIndex, FechaHora
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s04-ml-distribuido-regresion/data/variables_ambientales.csv
                                                                                

WindDir nula: 186,664 de 186,664 (100.0%)


In [10]:
errores_cm = df_sin_winddir.filter(col("Valor_CM") == 99999).count()
print(f"Filas con codigo de error Valor_CM=99999: {errores_cm:,}")

df_limpio = df_sin_winddir.filter(col("Valor_CM") != 99999)
print(f"Filas despues de eliminar el codigo de error: {df_limpio.count():,}")


26/09/03 02:32:52 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad., UVIndex, FechaHora
 Schema: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad, UVIndex, FechaHora
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s04-ml-distribuido-regresion/data/variables_ambientales.csv


Filas con codigo de error Valor_CM=99999: 2,126


26/09/03 02:32:54 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad., UVIndex, FechaHora
 Schema: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad, UVIndex, FechaHora
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s04-ml-distribuido-regresion/data/variables_ambientales.csv
                                                                                

Filas despues de eliminar el codigo de error: 184,538


## 3.3.2 Tratar nulos, errores y duplicados


Primero, confirmar que ni la deduplicacion ni el `join` dejaron `FechaHora` repetida:


In [11]:
total_final = df_limpio.count()
sin_duplicar = df_limpio.dropDuplicates(["FechaHora"]).count()

print(f"Total: {total_final:,}, sin duplicar por FechaHora: {sin_duplicar:,}")
assert total_final == sin_duplicar, "Hay FechaHora duplicada en la tabla integrada final"


26/09/03 02:32:57 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad., UVIndex, FechaHora
 Schema: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad, UVIndex, FechaHora
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s04-ml-distribuido-regresion/data/variables_ambientales.csv
[Stage 152:==========================================>              (3 + 1) / 4]

Total: 184,538, sin duplicar por FechaHora: 184,538


Con eso confirmado, quedan los nulos finales sobre las variables fisicas:


In [12]:
VARIABLES_9 = [
    "Valor_CE", "Valor_CM", "TempOut", "OutHum",
    "WindSpeed", "Bar", "Rain", "SolarRad", "UVIndex",
]

antes = df_limpio.count()
df_valido = df_limpio.na.drop(subset=VARIABLES_9)
despues = df_valido.count()

print(f"Filas antes: {antes:,}, despues de na.drop(subset=VARIABLES_9): {despues:,}")
print(f"Filas eliminadas por nulos en variables criticas: {antes - despues:,}")

df_valido = df_valido.cache()


26/09/03 02:33:02 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad., UVIndex, FechaHora
 Schema: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad, UVIndex, FechaHora
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s04-ml-distribuido-regresion/data/variables_ambientales.csv
26/09/03 02:33:05 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad., UVIndex, FechaHora
 Schema: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad, UVIndex, FechaHora
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s04-ml-distribuido-regresion/data/variables_ambientales.csv
                                                                                

Filas antes: 184,538, despues de na.drop(subset=VARIABLES_9): 184,538
Filas eliminadas por nulos en variables criticas: 0


Con la tabla ya limpia, se persiste como capa Gold particionada por mes, en vez de
dejarla solo en memoria:


In [13]:
from pyspark.sql.functions import date_format

ARTIFACTS = "/opt/s04-ml-distribuido-regresion/artifacts"

df_particionable = df_valido.withColumn("AnioMes", date_format(col("FechaHora"), "yyyy-MM"))

(
    df_particionable
    .repartition(4)
    .write.format("parquet")
    .mode("overwrite")
    .partitionBy("AnioMes")
    .save(f"{ARTIFACTS}/campo_electrico_particionado")
)

import os
for carpeta in sorted(os.listdir(f"{ARTIFACTS}/campo_electrico_particionado")):
    print(carpeta)


26/09/03 02:33:07 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad., UVIndex, FechaHora
 Schema: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad, UVIndex, FechaHora
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s04-ml-distribuido-regresion/data/variables_ambientales.csv
                                                                                

._SUCCESS.crc
AnioMes=2025-05
AnioMes=2025-06
AnioMes=2025-07
AnioMes=2025-08
AnioMes=2025-09
AnioMes=2025-10
AnioMes=2025-11
AnioMes=2025-12
_SUCCESS


Se lee de vuelta para confirmar que no hubo perdida de filas, y que Spark usa
`PartitionFilters` en el plan de ejecucion al filtrar por `AnioMes`:


In [14]:
df_verificacion = spark.read.parquet(f"{ARTIFACTS}/campo_electrico_particionado")
df_verificacion.printSchema()

assert df_verificacion.count() == df_particionable.count()
print(f"Verificado: {df_verificacion.count():,} filas, ida y vuelta sin perdida.")

df_verificacion.filter(col("AnioMes") == "2025-09").explain(True)


root
 |-- FechaHora: timestamp (nullable = true)
 |-- Valor_CE: double (nullable = true)
 |-- Valor_CM: double (nullable = true)
 |-- TempOut: double (nullable = true)
 |-- OutHum: double (nullable = true)
 |-- WindSpeed: double (nullable = true)
 |-- Bar: double (nullable = true)
 |-- Rain: double (nullable = true)
 |-- SolarRad: double (nullable = true)
 |-- UVIndex: double (nullable = true)
 |-- AnioMes: string (nullable = true)



Verificado: 184,538 filas, ida y vuelta sin perdida.
== Parsed Logical Plan ==
'Filter '`=`('AnioMes, 2025-09)
+- Relation [FechaHora#3368,Valor_CE#3369,Valor_CM#3370,TempOut#3371,OutHum#3372,WindSpeed#3373,Bar#3374,Rain#3375,SolarRad#3376,UVIndex#3377,AnioMes#3378] parquet

== Analyzed Logical Plan ==
FechaHora: timestamp, Valor_CE: double, Valor_CM: double, TempOut: double, OutHum: double, WindSpeed: double, Bar: double, Rain: double, SolarRad: double, UVIndex: double, AnioMes: string
Filter (AnioMes#3378 = 2025-09)
+- Relation [FechaHora#3368,Valor_CE#3369,Valor_CM#3370,TempOut#3371,OutHum#3372,WindSpeed#3373,Bar#3374,Rain#3375,SolarRad#3376,UVIndex#3377,AnioMes#3378] parquet

== Optimized Logical Plan ==
Filter (isnotnull(AnioMes#3378) AND (AnioMes#3378 = 2025-09))
+- Relation [FechaHora#3368,Valor_CE#3369,Valor_CM#3370,TempOut#3371,OutHum#3372,WindSpeed#3373,Bar#3374,Rain#3375,SolarRad#3376,UVIndex#3377,AnioMes#3378] parquet

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parq

Para ver cuantas filas quedaron guardadas en cada particion, sin salir de Spark ni
contar archivos a mano:


In [15]:
df_verificacion.groupBy("AnioMes").count().orderBy("AnioMes").show(truncate=False)


+-------+-----+
|AnioMes|count|
+-------+-----+
|2025-05|28107|
|2025-06|41502|
|2025-07|1409 |
|2025-08|20582|
|2025-09|27397|
|2025-10|32877|
|2025-11|32652|
|2025-12|12   |
+-------+-----+



In [16]:
df_valido.unpersist()


DataFrame[FechaHora: timestamp, Valor_CE: double, Valor_CM: double, TempOut: double, OutHum: double, WindSpeed: double, Bar: double, Rain: double, SolarRad: double, UVIndex: double]

`df_verificacion` ya es la salida Gold, leida y verificada — no hace falta volver a leer
el Parquet desde disco para empezar la parte de modelado:


In [17]:
df = df_verificacion

df.printSchema()
print(f"Filas: {df.count():,}")
df.describe(VARIABLES_9).show()


root
 |-- FechaHora: timestamp (nullable = true)
 |-- Valor_CE: double (nullable = true)
 |-- Valor_CM: double (nullable = true)
 |-- TempOut: double (nullable = true)
 |-- OutHum: double (nullable = true)
 |-- WindSpeed: double (nullable = true)
 |-- Bar: double (nullable = true)
 |-- Rain: double (nullable = true)
 |-- SolarRad: double (nullable = true)
 |-- UVIndex: double (nullable = true)
 |-- AnioMes: string (nullable = true)

Filas: 184,538


[Stage 208:====================>                                   (4 + 7) / 11]

+-------+-------------------+------------------+------------------+-----------------+-----------------+-----------------+--------------------+------------------+------------------+
|summary|           Valor_CE|          Valor_CM|           TempOut|           OutHum|        WindSpeed|              Bar|                Rain|          SolarRad|           UVIndex|
+-------+-------------------+------------------+------------------+-----------------+-----------------+-----------------+--------------------+------------------+------------------+
|  count|             184538|            184538|            184538|           184538|           184538|           184538|              184538|            184538|            184538|
|   mean|-0.9489558248165707|24278.279628585944|17.549393078931907|82.30233881368606|3.911056801309197|949.1927304945158|2.167575241955586...|174.62464641428866|1.2261539628694533|
| stddev| 0.8908501045924285|60.188433959341104| 3.008674476530922|7.323218566900627|4.87647261

## De esta salida a dos preguntas distintas: regresion (S4) y series de tiempo (S10)

`campo_electrico_particionado/` no tiene un solo destino. La misma tabla alimenta dos
sesiones que le hacen a los datos preguntas de naturaleza distinta:

| | S4 — Regresion (hoy) | S10 — Series de tiempo |
|---|---|---|
| Pregunta de fondo | Que otras variables explican `Valor_CE`? | El pasado de `Valor_CE` predice su futuro? |
| Predictores | Las otras 8 variables, en el mismo instante `t` | `Valor_CE` en instantes anteriores (`t`, `t-1`, ...) |
| Objetivo | `Valor_CE` en ese mismo instante `t` | `Valor_CE` en un instante futuro (`t+1`) |
| Orden de las filas | Irrelevante — cada fila es independiente | Critico — el orden cronologico es el dato |
| Division entrenamiento/prueba | Aleatoria (`randomSplit`) | Cronologica (equivalente a `TimeSeriesSplit`) |
| Riesgo si se usa la division del otro caso | Ninguno | Fuga de informacion: el modelo "veria" el futuro al entrenar |


## 3.3.3 Seleccionar predictores


In [18]:
PREDICTORES = [v for v in VARIABLES_9 if v != "Valor_CE"]
print(f"Predictores ({len(PREDICTORES)}): {PREDICTORES}")


Predictores (8): ['Valor_CM', 'TempOut', 'OutHum', 'WindSpeed', 'Bar', 'Rain', 'SolarRad', 'UVIndex']


## 3.3.4 Ensamblar el vector de predictores (`VectorAssembler`)


In [19]:
from pyspark.ml.feature import VectorAssembler

ensamblador = VectorAssembler(inputCols=PREDICTORES, outputCol="features")
df_ml = ensamblador.transform(df).select("features", "Valor_CE")

df_ml.show(5, truncate=False)


+--------------------------------------------+--------+
|features                                    |Valor_CE|
+--------------------------------------------+--------+
|[24327.1,20.9,77.0,11.3,949.6,0.0,349.0,2.0]|-2.17   |
|[24274.8,16.1,87.0,4.8,949.4,0.0,184.0,1.5] |-3.02   |
|[24250.4,19.6,83.0,4.8,952.0,0.0,411.0,2.1] |-0.27   |
|[24280.4,15.2,88.0,0.0,950.8,0.0,0.0,0.0]   |0.12    |
|[24278.1,13.8,84.0,3.2,950.8,0.0,0.0,0.0]   |-0.86   |
+--------------------------------------------+--------+
only showing top 5 rows


## 3.3.5 Dividir aleatoriamente en entrenamiento y prueba


In [20]:
df_train, df_test = df_ml.randomSplit([0.8, 0.2], seed=42)

print(f"Entrenamiento: {df_train.count():,} filas")
print(f"Prueba: {df_test.count():,} filas")


Entrenamiento: 147,943 filas
Prueba: 36,595 filas


## 3.3.6 Escalar si aplica


"Si aplica" es literal aqui, y en Spark MLlib no aplica: `LinearRegression` tiene el
parametro `standardization` (`True` por defecto) — ya estandariza los predictores
internamente antes de ajustar el modelo, precisamente para que `regParam` (3.4.3) penalice
de forma justa entre variables de escalas muy distintas (`Valor_CM` en miles, `Rain` entre
0 y 0.2), y devuelve los coeficientes en la escala original de `features`, no en unidades
estandarizadas (documentacion oficial de Spark). Escalar a mano con `StandardScaler` antes
de entrenar seria trabajo redundante. `RandomForestRegressor` (3.4.4) tampoco lo necesita
— sus arboles dividen por umbrales, no por magnitud de coeficientes.

Este paso no siempre "no aplica": en librerias que no estandarizan internamente, escalar a
mano sigue siendo necesario — vale la pena confirmarlo en la documentacion del modelo
concreto, no asumirlo por costumbre. Por eso no hay celda de codigo aqui: no hace falta
ninguna transformacion adicional, `df_train`/`df_test` siguen igual.


## Fase 4 — Modeling


## 3.4.1 Entrenar regresion lineal


In [21]:
from pyspark.ml.regression import LinearRegression

lr_base = LinearRegression(featuresCol="features", labelCol="Valor_CE")
modelo_base = lr_base.fit(df_train)

print("Coeficientes:", modelo_base.coefficients)
print("Intercepto:", modelo_base.intercept)


26/09/03 02:33:35 WARN Instrumentation: [61225304] regParam is zero, which might cause numerical instability and overfitting.
netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory
netlib-lapack: JNI_OnLoad: dlopen(liblapack.so.3) failed: liblapack.so.3: cannot open shared object file: No such file or directory
                                                                                

Coeficientes: [-0.002788944930500952,0.145401072923687,0.006895259414812994,-0.07792759472452553,-0.04286953466350843,1.2398731396808425,-0.0007062868107990013,0.041606624500483594]
Intercepto: 104.71155302801812


**Advertencias esperadas, no errores**: pueden aparecer `regParam is zero, which might
cause numerical instability and overfitting` (es la linea base a proposito, sin regularizar)
y `netlib-blas: JNI_OnLoad...` (falta una libreria nativa, Spark usa JVM pura — no afecta
resultados).


## 3.4.2 Evaluar con RMSE, R2 y MAE


In [22]:
from pyspark.ml.evaluation import RegressionEvaluator

predicciones_base = modelo_base.transform(df_test)
predicciones_base.select("Valor_CE", "prediction").show(5)

def evaluar(predicciones, nombre):
    resultados = {}
    for metrica in ["rmse", "r2", "mae"]:
        evaluador = RegressionEvaluator(labelCol="Valor_CE", predictionCol="prediction", metricName=metrica)
        resultados[metrica.upper()] = evaluador.evaluate(predicciones)
    print(f"{nombre}: RMSE={resultados['RMSE']:.4f}  R2={resultados['R2']:.4f}  MAE={resultados['MAE']:.4f}")
    return resultados

resultados_base = evaluar(predicciones_base, "LinearRegression base")


+--------+--------------------+
|Valor_CE|          prediction|
+--------+--------------------+
|   -2.24|-0.19094105242808723|
|   -1.88|-0.16048159803641227|
|   -2.32| -0.2944109093496792|
|   -2.42| -0.3203480972033361|
|   -2.08|-0.01823941523674...|
+--------+--------------------+
only showing top 5 rows


[Stage 231:=========================>                              (5 + 6) / 11]

LinearRegression base: RMSE=0.7909  R2=0.2271  MAE=0.6087


## 3.4.3 Probar regularizacion e hiperparametros basicos


In [23]:
configuraciones = [
    {"nombre": "Sin regularizacion", "regParam": 0.0, "elasticNetParam": 0.0},
    {"nombre": "Ridge (L2)", "regParam": 0.1, "elasticNetParam": 0.0},
    {"nombre": "Elastic Net (L1+L2)", "regParam": 0.1, "elasticNetParam": 0.5},
]

comparacion_configs = []
for config in configuraciones:
    lr = LinearRegression(
        featuresCol="features", labelCol="Valor_CE",
        regParam=config["regParam"], elasticNetParam=config["elasticNetParam"],
    )
    modelo = lr.fit(df_train)
    predicciones = modelo.transform(df_test)
    resultado = evaluar(predicciones, config["nombre"])
    resultado["Configuracion"] = config["nombre"]
    comparacion_configs.append(resultado)

import pandas as pd
pd.DataFrame(comparacion_configs)[["Configuracion", "RMSE", "R2", "MAE"]]


26/09/03 02:33:46 WARN Instrumentation: [465d8099] regParam is zero, which might cause numerical instability and overfitting.
                                                                                

Sin regularizacion: RMSE=0.7909  R2=0.2271  MAE=0.6087


Ridge (L2): RMSE=0.7962  R2=0.2166  MAE=0.6174


Elastic Net (L1+L2): RMSE=0.8091  R2=0.1910  MAE=0.6372


,Configuracion,RMSE,R2,MAE
0,Sin regularizacion,0.790852,0.227109,0.608656
1,Ridge (L2),0.796224,0.216575,0.617418
2,Elastic Net (L1+L2),0.809133,0.190964,0.637151


La fila "Sin regularizacion" es una verificacion util: deberia salir igual al modelo
base de 3.4.2 — si no coincide, algo cambio entre celdas.


## 3.4.4 Entrenar un modelo basado en arboles


In [24]:
from pyspark.ml.regression import RandomForestRegressor

rf = RandomForestRegressor(featuresCol="features", labelCol="Valor_CE", numTrees=50, maxDepth=8, seed=42)
modelo_rf = rf.fit(df_train)


26/09/03 02:34:12 WARN DAGScheduler: Broadcasting large task binary with size 1029.6 KiB
26/09/03 02:34:14 WARN DAGScheduler: Broadcasting large task binary with size 1963.6 KiB
                                                                                

## 3.4.5 Evaluar con RMSE, R2 y MAE


In [25]:
predicciones_rf = modelo_rf.transform(df_test)

resultados_rf = evaluar(predicciones_rf, "Random Forest")


[Stage 305:==========>                                             (2 + 9) / 11]

Random Forest: RMSE=0.6732  R2=0.4399  MAE=0.5001


## 3.4.6 Analizar importancia de variables


`RandomForestRegressor` calcula, sin costo adicional, `featureImportances`: una proporcion
de cuanto reduce cada variable el error del modelo en promedio — las proporciones de las 8
variables suman 1.0.


In [26]:
importancias = list(zip(PREDICTORES, modelo_rf.featureImportances.toArray()))
importancias.sort(key=lambda x: x[1], reverse=True)

for variable, importancia in importancias:
    print(f"{variable:12s} {importancia:.4f}")


WindSpeed    0.3365
TempOut      0.2023
OutHum       0.1564
Valor_CM     0.1262
SolarRad     0.0800
Bar          0.0558
UVIndex      0.0428
Rain         0.0000


## Fase 5 — Evaluation


## 3.5.1 Comparar los modelos candidatos


In [27]:
comparacion_final = pd.DataFrame(comparacion_configs + [
    {**resultados_rf, "Configuracion": "Random Forest"}
])[["Configuracion", "RMSE", "R2", "MAE"]]

comparacion_final.sort_values("RMSE")


,Configuracion,RMSE,R2,MAE
3,Random Forest,0.673222,0.439928,0.500095
0,Sin regularizacion,0.790852,0.227109,0.608656
1,Ridge (L2),0.796224,0.216575,0.617418
2,Elastic Net (L1+L2),0.809133,0.190964,0.637151


## 3.5.2 Seleccionar el mejor modelo


Con base en la tabla de la celda anterior, elige cual configuracion tuvo el mejor RMSE en
tu propia corrida. El nombre de variable `modelo_ganador` de 3.6.1 asume que fue Random
Forest — ajustalo segun tu resultado real.


## 3.5.3 Validar si el error es aceptable para el negocio


El alcance declarado en 3.1.3 era comparar cuatro configuraciones y reportar RMSE, R2 y
MAE — eso ya se cumplio. Sobre el desempeño en si: compara el R2 del ganador contra el
umbral que exigiria un sistema en produccion real, y documenta tu conclusion aqui.


## Fase 6 — Deployment


## 3.6.1 Guardar el modelo seleccionado


In [28]:
modelo_ganador = modelo_rf  # ajusta esta linea segun tu propio resultado (3.5.2)

modelo_ganador.write().overwrite().save(f"{ARTIFACTS}/modelo_ce_regresion")
print(f"Modelo guardado en {ARTIFACTS}/modelo_ce_regresion")


Modelo guardado en /opt/s04-ml-distribuido-regresion/artifacts/modelo_ce_regresion


## Cierre


## 3.7.1 Documentar hallazgos y responder preguntas de reflexion


Agrega celdas markdown breves debajo de cada bloque de codigo relevante explicando que
hiciste y que observaste.

**Reflexion tecnica breve** (5 a 8 lineas): por que resolver los duplicados de variables
ambientales antes del `join` evita un problema mas dificil de rastrear despues? que
configuracion tuvo el mejor RMSE, y por cuanto margen supero a la linea base? cual fue la
variable con mayor `featureImportances`, y tiene sentido fisico? por que `VectorAssembler`
es un paso obligatorio en Spark MLlib y no en scikit-learn?
